# 🥗 Nutrition RAG — Full Pipeline (v3)
**Architecture:** Data Ingestion (VN-enriched) → Language Detection → Query Expansion (VI→EN) → Query Preprocessing → Hybrid Retrieval → Strict Bilingual Generation → Evaluation

### v3 Improvements
1. **Song ngữ:** Phát hiện ngôn ngữ câu hỏi → trả lời hoàn toàn bằng tiếng Việt hoặc tiếng Anh; giữ tên thực phẩm song ngữ khi hữu ích.
2. **Truy vấn tiếng Việt:** Enrich index + mở rộng query (lexicon + LLM) để tra "thịt bò" → `Beef, grade I`.
3. **Chống OOM trên T4:** Model orchestrator, embed trên CPU khi inference, reranker lazy-load, giới hạn context/batch.

> Chạy từng section theo thứ tự. GPU T4 (16GB VRAM) được tối ưu.


In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
import os, shutil

ROOT_DIR = "/content/drive/MyDrive/nutrition_rag"
PROCESSED_DIR = os.path.join(ROOT_DIR, "data_processed")
os.makedirs(PROCESSED_DIR, exist_ok=True)
ALL_RECORDS_FILE = os.path.join(PROCESSED_DIR, "all_records.json")

if not os.path.exists(ALL_RECORDS_FILE):
    from google.colab import files
    print("Chọn file all_records.json để upload...")
    uploaded = files.upload()
    src = next(iter(uploaded))
    shutil.move(src, ALL_RECORDS_FILE)
    print(f"✅ Đã lưu → {ALL_RECORDS_FILE}")
else:
    print(f"✅ Đã có sẵn: {ALL_RECORDS_FILE}")

✅ Đã có sẵn: /content/drive/MyDrive/nutrition_rag/data_processed/all_records.json


In [3]:
# Core
!pip install -q sentence-transformers chromadb rank_bm25

# Fix version cho 4-bit models
!pip install -q transformers==4.44.2
!pip install -q accelerate==0.34.2
!pip install -q bitsandbytes==0.43.3

# Retrieval
!pip install -q FlagEmbedding

# Evaluation
!pip install -q ragas datasets openai

# Metrics
!pip install -q rouge-score nltk scikit-learn langdetect

print("✅ Đã cài đặt xong tất cả thư viện.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 83.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 89.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 93.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 5.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the s

In [4]:
import gc, os, json, re, torch, numpy as np
from langdetect import detect, LangDetectException

VRAM_CONFIG = {
    "embed_device_indexing": "cuda",
    "embed_device_inference": "cpu",
    "reranker_lazy": True,
    "rerank_batch_size": 8,
    "rrf_pool": 30,
    "embed_batch_size": 64,
    "max_context_chars": 2400,
    "max_doc_chars": 500,
    "use_hyde_for_lookup": False,
    "collection_name": "nutrition_collection_v3",
}

def clear_vram():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def print_vram(label=""):
    if torch.cuda.is_available():
        used = torch.cuda.memory_allocated() / 1024**3
        reserved = torch.cuda.memory_reserved() / 1024**3
        print(f"  VRAM [{label}]: allocated={used:.2f}GB, reserved={reserved:.2f}GB")
    else:
        print(f"  [{label}] CUDA không khả dụng — dùng CPU.")


In [5]:
import re

# ── Language Detection ───────────────────────────────────────────────────────
def detect_language(text: str) -> str:
    """Detect nếu text là tiếng Việt dựa trên ký tự đặc trưng (dấu, đ)."""
    vn_pattern = r'[àáảãạăắằẳẵặâấầẩẫậèéẻẽẹêếềểễệìíỉĩịòóỏõọôốồổỗộơớờởỡợùúủũụưứừửữựỳýỷỹỵđ]'
    if re.search(vn_pattern, text, re.IGNORECASE):
        return "vi"
    return "en"

# ── LLM Translation ──────────────────────────────────────────────────────────
def translate_query_llm(query: str) -> str:
    """
    Dùng LLM dịch toàn bộ query Tiếng Việt sang Tiếng Anh để giữ nguyên ngữ cảnh.
    """
    system = (
        "You are an expert translator. Translate the following Vietnamese nutrition query to English. "
        "Keep food names and context accurate. Only return the English translation, nothing else."
    )
    return llm_call(system, query, max_new_tokens=64, temperature=0.1)

# Smoke test
print(f"detect_language('Thịt bò có bao nhiêu protein?') = {detect_language('Thịt bò có bao nhiêu protein?')}")
print(f"translate_query_llm('Trứng vịt lộn có bao nhiêu calo?') = {translate_query_llm('Trứng vịt lộn có bao nhiêu calo?')}")
print(f"detect_language('How many calories are in scrambled duck eggs?') = {detect_language('How many calories are in scrambled duck eggs?')}")
print(f"translate_query_llm('How many calories are in scrambled duck eggs?') = {translate_query_llm('How many calories are in scrambled duck eggs?')}")

detect_language('Thịt bò có bao nhiêu protein?') = vi


NameError: name 'llm_call' is not defined

## 🗄️ 1. Tầng 1 — Data Ingestion
Load knowledge base → BAAI/bge-m3 embedding (**on CPU** để tiết kiệm VRAM) → ChromaDB + BM25 Index


In [6]:
import os, json, torch, numpy as np
import chromadb
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
from tqdm import tqdm

# ── Đường dẫn ─────────────────────────────────────────────────────────────────────
ROOT_DIR       = "/content/drive/MyDrive/nutrition_rag"
PROCESSED_DIR  = os.path.join(ROOT_DIR, "data_processed")
VECTOR_DB_DIR  = os.path.join(ROOT_DIR, "vector_db_v3")   # <-- V3 DB riêng
ALL_RECORDS_FILE = os.path.join(PROCESSED_DIR, "all_records.json")
OUTPUT_DIR     = os.path.join(ROOT_DIR, "outputs")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Load Knowledge Base ─────────────────────────────────────────────────────
with open(ALL_RECORDS_FILE, "r", encoding="utf-8") as f:
    all_records = json.load(f)
print(f"Đã load {len(all_records)} records từ all_records.json")

ids       = [r["record_id"]          for r in all_records]
documents = [r["text_for_embedding"] for r in all_records]

# ── Embedding Model: BAAI/bge-m3 (trên CPU để tiết kiệm VRAM cho Llama-3) ───
print("Đang tải BAAI/bge-m3 lên CPU (tiết kiệm ~2.3GB VRAM)...")
embed_model = SentenceTransformer("BAAI/bge-m3", device="cpu")
print("✅ BGE-M3 sẵn sàng trên CPU.")

# ── ChromaDB ─────────────────────────────────────────────────────────────────
client = chromadb.PersistentClient(path=VECTOR_DB_DIR)
collection = client.get_or_create_collection(
    name="nutrition_collection_v3",
    metadata={"hnsw:space": "cosine"}
)

existing_count = collection.count()
if existing_count > 0:
    print(f"ChromaDB đã có {existing_count} bản ghi → Bỏ qua re-embedding.")
else:
    print("ChromaDB trống → Bắt đầu embedding (trên CPU, sẽ chậm hơn GPU ~3x)...")
    metadatas = []
    for record in all_records:
        meta = {
            "record_type": record.get("record_type", ""),
            "source":      record.get("source", ""),
        }
        for key in ("group", "age", "food_name_en"):
            if record.get(key):
                meta[key] = record[key]
        metadatas.append(meta)

    BATCH = 64  # Batch nhỏ hơn cho CPU
    for i in tqdm(range(0, len(ids), BATCH), desc="Embedding"):
        s, e = i, i + BATCH
        batch_embeddings = embed_model.encode(
            documents[s:e], convert_to_tensor=False,
            show_progress_bar=False
        ).tolist()
        collection.upsert(
            ids=ids[s:e],
            documents=documents[s:e],
            embeddings=batch_embeddings,
            metadatas=metadatas[s:e]
        )
    print(f"✅ Đã tạo {collection.count()} vectors vào ChromaDB.")

# ── BM25 Index ──────────────────────────────────────────────────────────────
print("Đang khởi tạo BM25 Index...")
tokenized_corpus = [doc.lower().split() for doc in documents]
bm25 = BM25Okapi(tokenized_corpus)
print("✅ BM25 Index sẵn sàng.")


Đã load 583 records từ all_records.json
Đang tải BAAI/bge-m3 lên CPU (tiết kiệm ~2.3GB VRAM)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

✅ BGE-M3 sẵn sàng trên CPU.
ChromaDB đã có 583 bản ghi → Bỏ qua re-embedding.
Đang khởi tạo BM25 Index...
✅ BM25 Index sẵn sàng.


## 🔧 2. Tầng 2 — Query Preprocessing
### 2a. Load Local Llama-3-8B-Instruct (Unsloth 4-bit)


In [7]:
from transformers import pipeline as hf_pipeline

print("Đang tải Llama-3-8B-Instruct (4-bit)...")

local_llm = hf_pipeline(
    "text-generation",
    model="unsloth/llama-3-8b-Instruct-bnb-4bit",
    device_map="auto",
)

print("✅ Llama-3 sẵn sàng trên GPU.")

def llm_call(system_prompt: str, user_prompt: str,
             max_new_tokens: int = 256, temperature: float = 0.1) -> str:
    """Helper gọi local Llama-3 với temperature thấp."""
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": user_prompt},
    ]
    prompt = local_llm.tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    outputs = local_llm(
        prompt,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        do_sample=True,
        return_full_text=False,
    )
    return outputs[0]["generated_text"].strip()


Đang tải Llama-3-8B-Instruct (4-bit)...


config.json: 0.00B [00:00, ?B/s]

Unused kwargs: ['_load_in_4bit', '_load_in_8bit', 'quant_method']. These kwargs are not used in <class 'transformers.utils.quantization_config.BitsAndBytesConfig'>.


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

RuntimeError: Failed to import transformers.integrations.bitsandbytes because of the following error (look up to see its traceback):
No module named 'triton.ops'

### 2b. Block 1 — Query Router (Bilingual)

In [ ]:
ROUTE_CONFIG = {
    "LOOKUP":    {"top_k": 3},
    "COMPARE":   {"top_k": 6},
    "RECOMMEND": {"top_k": 10},
    "EXPLAIN":   {"top_k": 5},
}

def route_query(query: str) -> str:
    """Phân loại intent câu hỏi: LOOKUP | COMPARE | RECOMMEND | EXPLAIN (bilingual)"""
    lang = detect_language(query)
    if lang == "vi":
        system = (
            "Bạn là bộ định tuyến hệ thống. Phân loại câu hỏi vào ĐÚNG 1 trong 4 loại:\n"
            "- LOOKUP: tra cứu số liệu dinh dưỡng cụ thể của 1 thực phẩm.\n"
            "- COMPARE: so sánh 2+ thực phẩm.\n"
            "- RECOMMEND: gợi ý thực đơn / chế độ ăn.\n"
            "- EXPLAIN: giải thích khái niệm dinh dưỡng.\n"
            "Chỉ trả về 1 từ duy nhất, không giải thích."
        )
    else:
        system = (
            "You are a system router. Classify the query into EXACTLY 1 of 4 types:\n"
            "- LOOKUP: looking up specific nutrition data of 1 food.\n"
            "- COMPARE: comparing 2+ foods.\n"
            "- RECOMMEND: suggesting a diet or meal plan.\n"
            "- EXPLAIN: explaining a nutrition concept.\n"
            "Return only 1 word, no explanation."
        )
    raw = llm_call(system, f'Query: "{query}"', max_new_tokens=10, temperature=0.1)
    for intent in ["LOOKUP", "COMPARE", "RECOMMEND", "EXPLAIN"]:
        if intent in raw.upper():
            return intent
    return "LOOKUP"

# Smoke test
print(route_query("Thịt bò grade I có bao nhiêu protein?"))       # LOOKUP
print(route_query("Compare protein in beef and chicken"))   # COMPARE


LOOKUP
COMPARE


### 2c. Block 2 — Query Decomposition (Bilingual)

In [ ]:
def decompose_query(query: str) -> list[str]:
    """
    Nếu câu hỏi phức tạp / so sánh, tách thành các sub-query độc lập.
    Trả về list[str] — câu đơn thì list 1 phần tử. Bilingual support.
    """
    lang = detect_language(query)
    if lang == "vi":
        system = (
            "Bạn là hệ thống tách câu hỏi. Nếu câu hỏi yêu cầu so sánh hoặc hỏi nhiều thực phẩm, "
            "hãy tách thành các câu hỏi con độc lập, mỗi câu 1 thực phẩm. "
            "Nếu câu hỏi đã đơn giản, trả về đúng câu hỏi gốc. "
            "Trả về kết quả dạng danh sách, mỗi câu 1 dòng, không đánh số, không giải thích thêm."
        )
    else:
        system = (
            "You are a query decomposition system. If the query compares or asks about multiple foods, "
            "split it into independent sub-queries, one per food. "
            "If the query is already simple, return it as-is. "
            "Return a list, one query per line, no numbering, no explanation."
        )
    raw = llm_call(system, f'Query: "{query}"', max_new_tokens=128, temperature=0.1)
    sub_queries = [line.strip("- •").strip() for line in raw.splitlines() if line.strip()]
    # Loại dòng rỗng và giữ tối đa 5 sub-query
    sub_queries = [q for q in sub_queries if len(q) > 5][:5]
    return sub_queries if sub_queries else [query]

# Test
test_q = "So sánh lượng đạm trong thịt bò loại I và ức gà"
print(decompose_query(test_q))


['lượng đạm trong thịt bò loại I', 'lượng đạm trong ức gà']


### 2d. Block 3 — HyDE (Hypothetical Document Embeddings) — Bilingual

In [ ]:
def generate_hyde_doc(query: str) -> str:
    """
    Sinh câu trả lời 'nháp' từ kiến thức nội tại của Llama-3.
    Luôn sinh bằng tiếng Anh để match với text_for_embedding (English corpus).
    """
    # V3: Luôn sinh HyDE bằng English vì corpus là English
    system = (
        "You are a nutrition expert. Write a short paragraph (~50 words) in English "
        "describing the typical nutritional composition of the food being asked about. "
        "Include calories, protein, fat, carbohydrate values if possible. "
        "Only write the description, do not ask questions."
    )
    return llm_call(system, query, max_new_tokens=100, temperature=0.1)

def get_hybrid_query_embedding(query: str, use_hyde: bool = True):
    """
    Tạo embedding kết hợp câu hỏi gốc + HyDE document.
    Trả về vector trung bình để dùng cho Dense Search.
    """
    q_emb = embed_model.encode(query, convert_to_tensor=False)
    if use_hyde:
        hyde_doc = generate_hyde_doc(query)
        h_emb    = embed_model.encode(hyde_doc, convert_to_tensor=False)
        combined = (np.array(q_emb) + np.array(h_emb)) / 2.0
        return combined.tolist(), hyde_doc
    return q_emb.tolist(), None

# Test HyDE
emb, hyp_doc = get_hybrid_query_embedding("Cơm trắng 100g bao nhiêu calo?")
print(f"HyDE document:\n{hyp_doc}")
print(f"Embedding dim: {len(emb)}")


HyDE document:
Cơm trắng 100克 typically contains approximately 113 calories, with around 2. 8克protein, 0. 9克fat, and 24. 6克carbohydrates per 100克.
Embedding dim: 1024


## 🔍 3. Tầng 3 — Advanced Information Retrieval
### 3a. RRF Fusion Helper


In [ ]:
def reciprocal_rank_fusion(dense_ids: list, sparse_ids: list, k: int = 60) -> list:
    """Trộn kết quả Dense + Sparse bằng công thức RRF."""
    scores = {}
    for rank, doc_id in enumerate(dense_ids):
        scores[doc_id] = scores.get(doc_id, 0) + 1 / (k + rank + 1)
    for rank, doc_id in enumerate(sparse_ids):
        scores[doc_id] = scores.get(doc_id, 0) + 1 / (k + rank + 1)
    return sorted(scores.items(), key=lambda x: x[1], reverse=True)


### 3b. Cross-Encoder Reranker (bge-reranker-base) — CPU + Giới hạn candidates

In [ ]:
from FlagEmbedding import FlagReranker

print("Đang tải bge-reranker-base (trên CPU để tiết kiệm VRAM)...")

# V3: Load reranker trên CPU để dành VRAM cho Llama-3
reranker = FlagReranker("BAAI/bge-reranker-base", use_fp16=True, device="cpu")
print("✅ Reranker sẵn sàng trên CPU.")

# V3: Giới hạn số candidates cho reranker để giảm tải
MAX_RERANK_CANDIDATES = 20

def rerank_documents(query: str, candidate_ids: list, top_n: int = 5) -> list:
    """
    Nhận vào Top candidates từ RRF, chấm điểm lại từng cặp (query, doc)
    bằng Cross-Encoder, trả về Top-N IDs chính xác nhất.
    V3: Giới hạn MAX_RERANK_CANDIDATES để tránh OOM.
    """
    if not candidate_ids:
        return []

    # V3: Cắt candidates xuống tối đa MAX_RERANK_CANDIDATES
    candidate_ids = candidate_ids[:MAX_RERANK_CANDIDATES]

    # Kéo nội dung text của tất cả candidates từ ChromaDB
    result = collection.get(ids=candidate_ids)
    id_to_text = {
        result["ids"][i]: result["documents"][i]
        for i in range(len(result["ids"]))
    }

    pairs  = [(query, id_to_text[doc_id]) for doc_id in candidate_ids if doc_id in id_to_text]
    scores = reranker.compute_score(pairs)

    # Đảm bảo scores luôn là list (khi chỉ có 1 pair, FlagReranker trả về float)
    if isinstance(scores, (int, float)):
        scores = [scores]

    # Ghép lại (id, score) và sắp xếp
    id_score_pairs = sorted(
        zip([d for d in candidate_ids if d in id_to_text], scores),
        key=lambda x: x[1], reverse=True
    )
    return [doc_id for doc_id, _ in id_score_pairs[:top_n]]


Đang tải bge-reranker-base (trên CPU để tiết kiệm VRAM)...
✅ Reranker sẵn sàng trên CPU.


### 3c. Full Hybrid Search Pipeline (Dense + Sparse + RRF + Reranker) — V3 Dual-Query

In [ ]:
def hybrid_search(query: str, top_k: int = 5, use_hyde: bool = True,
                  rrf_pool: int = 50, rerank: bool = True) -> str:
    """
    V3 Pipeline retrieval (Đã sửa lỗi Translation Collision)
    """
    lang = detect_language(query)

    # Dịch toàn bộ query nếu là tiếng Việt
    if lang == "vi":
        en_query = translate_query_llm(query)
    else:
        en_query = query

    # 1. Dense search (BGE-M3)
    # Dùng en_query để tạo embedding khớp nhất với text_for_embedding (đang là tiếng Anh)
    query_emb, _hyde = get_hybrid_query_embedding(en_query, use_hyde=use_hyde)
    dense_resp = collection.query(
        query_embeddings=[query_emb],
        n_results=min(rrf_pool, collection.count())
    )
    dense_ids = dense_resp["ids"][0]

    # 2. Sparse search (BM25)
    # BM25 nhận câu tiếng Anh hoàn chỉnh (vd: "duck egg embryonated") để search
    tokenized_query = en_query.lower().split()
    bm25_scores     = bm25.get_scores(tokenized_query)
    sparse_top_idx  = np.argsort(bm25_scores)[::-1][:rrf_pool]
    sparse_ids      = [ids[i] for i in sparse_top_idx]

    # 3. RRF Fusion
    rrf_ranking     = reciprocal_rank_fusion(dense_ids, sparse_ids)
    candidate_ids   = [doc_id for doc_id, _ in rrf_ranking[:rrf_pool]]

    # 4. Reranker (Cross-Encoder)
    if rerank:
        # Reranker BGE đa ngôn ngữ tốt, có thể truyền thẳng query gốc tiếng Việt vào so sánh
        top_ids = rerank_documents(query, candidate_ids, top_n=top_k)
    else:
        top_ids = candidate_ids[:top_k]

    # 5. Build context string
    if not top_ids:
        return ""
    docs_data = collection.get(ids=top_ids)
    id_to_doc  = {docs_data["ids"][i]: (docs_data["documents"][i], docs_data["metadatas"][i])
                  for i in range(len(docs_data["ids"]))}
    context_parts = []
    for i, doc_id in enumerate(top_ids, 1):
        if doc_id in id_to_doc:
            text, meta = id_to_doc[doc_id]
            source = meta.get("source", "unknown")
            context_parts.append(f"--- Record {i} (ID: {doc_id} | Source: {source}) ---\n{text}")
    return "\n\n".join(context_parts)


    # Smoke test: tiếng Việt query phải tìm được Beef
print("=== Test VI query ===")
ctx = hybrid_search("Thịt bò có bao nhiêu protein?", top_k=3)
print(ctx[:1000])
print("\n=== Test EN query ===")
ctx_en = hybrid_search("How much protein in beef grade I?", top_k=3)
print(ctx_en[:1000])

=== Test VI query ===
--- Record 1 (ID: food_0282 | Source: food_composition_vn_2007) ---
Tên tiếng Việt: thịt bò
Food name: Beef, grade II
Calories: 167.0 kcal
Protein: 18.0 g
Fat: 10.5 g
Carbohydrate: 0.0 g
Fiber: 0.0 g
Calcium: 10.0 mg
Iron: 2.7 mg

--- Record 2 (ID: food_0283 | Source: food_composition_vn_2007) ---
Tên tiếng Việt: thịt bò
Food name: Beef, top loin, seperable lean only, trimmed to 1/8" fat, prime, raw
Calories: 127.0 kcal
Protein: 23.1 g
Fat: 3.9 g
Carbohydrate: 0.0 g
Fiber: 0.0 g
Calcium: 23.0 mg
Iron: 1.63 mg

--- Record 3 (ID: food_0329 | Source: food_composition_vn_2007) ---
Tên tiếng Việt: thịt bò
Food name: Beef, lung
Calories: 103.0 kcal
Protein: 15.2 g
Fat: 4.7 g
Carbohydrate: 0.0 g
Fiber: 0.0 g
Calcium: 10.0 mg
Iron: 6.7 mg

=== Test EN query ===
--- Record 1 (ID: food_0281 | Source: food_composition_vn_2007) ---
Tên tiếng Việt: thịt bò
Food name: Beef, grade I
Calories: 118.0 kcal
Protein: 21.0 g
Fat: 3.8 g
Carbohydrate: 0.0 g
Fiber: 0.0 g
Calcium: 12.0 mg